In [44]:
from keras.datasets import mnist 
from tensorflow.keras.utils import to_categorical
import numpy as np

np.random.seed(42)
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()



In [45]:
def preprocess_data(x, y, limit=100):
    all_selected_indices = []
    for i in range(10):
        index = np.where(y == i)[0][:limit]
        all_selected_indices.extend(index)

    all_selected_indices = np.random.permutation(all_selected_indices)
    x, y = x[all_selected_indices], y[all_selected_indices]

    x = x.reshape(len(x), 1, 28, 28)
    x = x.astype('float32') / 255
    y = to_categorical(y, num_classes=10)
    y = y.reshape(len(y), 1, 10)

    return x, y

x_train, y_train = preprocess_data(x_train_raw, y_train_raw)
x_test, y_test = preprocess_data(x_test_raw, y_test_raw)



In [ ]:


from convolutional import Convolutional
from relu import ReLU
from reshape import Reshape
from dense import Dense
from softmax import Softmax
from loss import categorical_cross_entropy, categorical_cross_entropy_prime

network = [
    Convolutional((1, 28, 28), 3, 5), 
    ReLU(), 
    Reshape((5, 26, 26), (1, 5 * 26 * 26)),
    Dense(5 * 26 * 26, 100),
    ReLU(),
    Dense(100, 10), 
    Softmax()
]

test_x = x_train[0]
out = test_x
for i, layer in enumerate(network):
    out = layer.forward(out)
    has_nan = np.any(np.isnan(out))
    print(f"Layer {i} ({type(layer).__name__}): shape={out.shape}, nan={has_nan}, max={np.nanmax(np.abs(out)):.4f}")

print(f"\nForward pass clean: {not np.any(np.isnan(out))}")
print("Network ready!")


Layer 0 (Convolutional): shape=(5, 26, 26), nan=False, max=0.6735
Layer 1 (ReLU): shape=(5, 26, 26), nan=False, max=0.3830
Layer 2 (Reshape): shape=(1, 3380), nan=False, max=0.3830
Layer 3 (Dense): shape=(1, 100), nan=False, max=0.8773
Layer 4 (ReLU): shape=(1, 100), nan=False, max=0.7337
Layer 5 (Dense): shape=(1, 10), nan=False, max=0.3501
Layer 6 (Softmax): shape=(1, 10), nan=False, max=0.1179

Forward pass clean: True
Network ready!


In [47]:
epochs = 20
lr = 0.01

for e in range(epochs):
    error = 0
    for x, y in zip(x_train, y_train):
        output = x
        for layer in network:
            output = layer.forward(output)

        error += categorical_cross_entropy(y, output)

        grad = categorical_cross_entropy_prime(y, output)
        for layer in reversed(network):
            grad = layer.backward(grad, lr)
    
    error /= len(x_train)
    print(f'{e+1}/{epochs}, error={error:.6f}')

1/20, error=1.049149
2/20, error=0.408285
3/20, error=0.225273
4/20, error=0.136853
5/20, error=0.085013
6/20, error=0.075229
7/20, error=0.012014
8/20, error=0.004353
9/20, error=0.002142
10/20, error=0.001511
11/20, error=0.001116
12/20, error=0.000916
13/20, error=0.000784
14/20, error=0.000683
15/20, error=0.000605
16/20, error=0.000543
17/20, error=0.000494
18/20, error=0.000451
19/20, error=0.000416
20/20, error=0.000386


In [48]:
print("Testing model on test set...")
correct = 0
total = len(x_test)

for i, (x, y) in enumerate(zip(x_test, y_test)):
    output = x
    # Forward pass through network
    for layer in network:
        output = layer.forward(output)
    
    pred = np.argmax(output)
    true = np.argmax(y)
    
    if pred == true:
        correct += 1
    
    # Print first 10 predictions as examples
    if i < 10:
        print(f'Sample {i+1}: pred={pred}, true={true}, {"✓" if pred == true else "✗"}')

print(f'\nFinal Results:')
print(f'Correct predictions: {correct}/{total}')
print(f'Accuracy: {correct/total:.2%}')

Testing model on test set...
Sample 1: pred=0, true=0, ✓
Sample 2: pred=6, true=6, ✓
Sample 3: pred=1, true=1, ✓
Sample 4: pred=5, true=5, ✓
Sample 5: pred=7, true=7, ✓
Sample 6: pred=9, true=9, ✓
Sample 7: pred=5, true=3, ✗
Sample 8: pred=6, true=6, ✓
Sample 9: pred=9, true=7, ✗
Sample 10: pred=2, true=2, ✓

Final Results:
Correct predictions: 875/1000
Accuracy: 87.50%


In [49]:
import numpy as np

weights = {}
for i, layer in enumerate(network):
    layer_name = f"{type(layer).__name__}_{i}"
    if hasattr(layer, 'weights'):       # Dense
        weights[f"{layer_name}_weights"] = layer.weights
        weights[f"{layer_name}_bias"] = layer.bias
    if hasattr(layer, 'kernels'):       # Convolutional
        weights[f"{layer_name}_kernels"] = layer.kernels
        weights[f"{layer_name}_biases"] = layer.biases

np.save("model_weights.npy", weights)
print("Model saved!")

Model saved!
